# AgentEdu Score

## 1. Setup e teste de conexão

In [1]:
!pip install basedosdados -q

import basedosdados as bd
from google.colab import auth
import pandas as pd
import numpy as np

auth.authenticate_user()

# Project ID do Google Cloud
PROJECT_ID = "marilsouza"

pd.set_option("display.max_columns", None)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 kB 7.0 MB/s eta 0:00:00


## 2–5. Escopo

Cinco cursos

`SINONIMOS_CURSO` existe porque o nome de "Ciências Sociais" aparece grafado de mais de uma forma
na base do INEP (com e sem acento)

In [2]:
COL_MATRICULAS = "quantidade_matriculas"
COL_INGRESSANTES = "quantidade_ingressantes"
COL_CONCLUINTES = "quantidade_concluintes"

UF = "PE"

CURSOS_DURACAO = {
    "Administração": 4,
    "Engenharia Civil": 5,
    "Direito": 5,
    "Pedagogia": 4,
    "Ciências Sociais": 4,
}

SINONIMOS_CURSO = {
    "Ciências Sociais": ["Ciências Sociais", "Ciencias Sociais"],
}

def nomes_curso(curso):
    """Retorna a lista de grafias possíveis do nome do curso na base do INEP."""
    return SINONIMOS_CURSO.get(curso, [curso])

# Anos de ingresso considerados (várias coortes deixam a taxa mais estável)
ANOS_INGRESSO = [2010, 2011, 2012]

# A janela de observação da conclusão vai até MULTIPLICADOR_JANELA vezes a duração
# padrão do curso — ex.: curso de 4 anos é observado até 8 anos depois do ingresso.
MULTIPLICADOR_JANELA = 2


## 6. Evasão de janela estendida (separando evasão real de atraso)

Ideia: em vez de olhar só "quem concluiu no prazo padrão", olhamos também "quem concluiu dentro de
uma janela mais larga" (o dobro da duração padrão do curso). A diferença entre as duas taxas é
**atraso** (terminou, só que depois do previsto) e não evasão. Só quem não aparece nem na janela
estendida é contado como **evasão real**.

Para cada curso e cada ano de ingresso:
- `ano_esperado = ano_ingresso + duracao` (prazo padrão)
- `ano_limite = ano_ingresso + duracao * MULTIPLICADOR_JANELA` (janela estendida)

E calculamos três taxas:
- `taxa_conclusao_no_prazo` = concluintes até `ano_esperado` / ingressantes
- `taxa_conclusao_ampliada` = concluintes até `ano_limite` (soma de todos os anos da janela) / ingressantes
- `taxa_atraso` = `taxa_conclusao_ampliada` − `taxa_conclusao_no_prazo`
- `taxa_evasao_real` = 1 − `taxa_conclusao_ampliada`

Mesma disciplina de sempre: `SUM` + `GROUP BY` **antes** do JOIN em cada CTE, para não sofrer o bug
de duplicação por turno/modalidade.

In [3]:
def calcular_evasao_janela(curso, duracao, ano_ingresso, uf=UF, multiplicador_janela=MULTIPLICADOR_JANELA):
    ano_esperado = ano_ingresso + duracao
    ano_limite = ano_ingresso + duracao * multiplicador_janela
    anos_janela = list(range(ano_esperado, ano_limite + 1))

    nomes = nomes_curso(curso)
    nomes_sql = ", ".join([f"'{n}'" for n in nomes])
    anos_janela_sql = ", ".join([str(a) for a in anos_janela])

    query = f"""
    WITH ingressantes AS (
      SELECT id_ies, sigla_uf,
             SUM({COL_INGRESSANTES}) AS ingressantes
      FROM `basedosdados.br_inep_censo_educacao_superior.curso`
      WHERE ano = {ano_ingresso}
        AND sigla_uf = '{uf}'
        AND nome_curso IN ({nomes_sql})
      GROUP BY id_ies, sigla_uf
    ),
    concluintes_no_prazo AS (
      SELECT id_ies, sigla_uf,
             SUM({COL_CONCLUINTES}) AS concluintes_no_prazo
      FROM `basedosdados.br_inep_censo_educacao_superior.curso`
      WHERE ano = {ano_esperado}
        AND sigla_uf = '{uf}'
        AND nome_curso IN ({nomes_sql})
      GROUP BY id_ies, sigla_uf
    ),
    concluintes_ampliado AS (
      SELECT id_ies, sigla_uf,
             SUM({COL_CONCLUINTES}) AS concluintes_ampliado
      FROM `basedosdados.br_inep_censo_educacao_superior.curso`
      WHERE ano IN ({anos_janela_sql})
        AND sigla_uf = '{uf}'
        AND nome_curso IN ({nomes_sql})
      GROUP BY id_ies, sigla_uf
    )
    SELECT
      i.id_ies,
      i.sigla_uf,
      i.ingressantes,
      COALESCE(p.concluintes_no_prazo, 0) AS concluintes_no_prazo,
      COALESCE(a.concluintes_ampliado, 0) AS concluintes_ampliado
    FROM ingressantes i
    LEFT JOIN concluintes_no_prazo p ON i.id_ies = p.id_ies AND i.sigla_uf = p.sigla_uf
    LEFT JOIN concluintes_ampliado a ON i.id_ies = a.id_ies AND i.sigla_uf = a.sigla_uf
    """

    df = bd.read_sql(query, billing_project_id=PROJECT_ID)
    df["nome_curso"] = curso
    df["ano_ingresso"] = ano_ingresso
    return df


dfs_evasao = []
for curso, duracao in CURSOS_DURACAO.items():
    for ano_ingresso in ANOS_INGRESSO:
        df_curso = calcular_evasao_janela(curso, duracao, ano_ingresso)
        dfs_evasao.append(df_curso)

df_evasao_bruto = pd.concat(dfs_evasao, ignore_index=True)
df_evasao_bruto = df_evasao_bruto[df_evasao_bruto["ingressantes"] > 0].copy()

df_evasao_bruto["taxa_conclusao_no_prazo"] = (
    df_evasao_bruto["concluintes_no_prazo"] / df_evasao_bruto["ingressantes"]
).clip(0, 1)
df_evasao_bruto["taxa_conclusao_ampliada"] = (
    df_evasao_bruto["concluintes_ampliado"] / df_evasao_bruto["ingressantes"]
).clip(0, 1)
df_evasao_bruto["taxa_atraso"] = (
    df_evasao_bruto["taxa_conclusao_ampliada"] - df_evasao_bruto["taxa_conclusao_no_prazo"]
).clip(0, 1)
df_evasao_bruto["taxa_evasao_real"] = (
    1 - df_evasao_bruto["taxa_conclusao_ampliada"]
).clip(0, 1)

print(df_evasao_bruto.shape)
df_evasao_bruto.groupby("nome_curso")[
    ["taxa_conclusao_no_prazo", "taxa_atraso", "taxa_evasao_real"]
].mean()


Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
(437, 11)


,taxa_conclusao_no_prazo,taxa_atraso,taxa_evasao_real
nome_curso,,,
Administração,0.463553,0.456649,0.079798
Ciências Sociais,0.193545,0.473122,0.333333
Direito,0.612607,0.387393,0.0
Engenharia Civil,0.622166,0.361666,0.016168
Pedagogia,0.555972,0.361451,0.082577


## 7. Consolidar a evasão por curso/IES

Uma linha por `id_ies + nome_curso`, com as taxas médias entre as coortes de ingresso usadas.

In [4]:
df_evasao = (
    df_evasao_bruto
    .groupby(["id_ies", "nome_curso", "sigla_uf"], as_index=False)
    .agg(
        taxa_evasao_real=("taxa_evasao_real", "mean"),
        taxa_atraso=("taxa_atraso", "mean"),
        ingressantes_total=("ingressantes", "sum"),
    )
)

# Removendo linhas de baixo volume (ruído estatístico de IES muito pequenas)
df_evasao = df_evasao[df_evasao["ingressantes_total"] >= 10].copy()

print(df_evasao.shape)
df_evasao.sort_values("taxa_evasao_real", ascending=False).head(10)


(156, 6)


,id_ies,nome_curso,sigla_uf,taxa_evasao_real,taxa_atraso,ingressantes_total
27,146,Administração,PE,1.0,0.0,14
28,1461,Pedagogia,PE,1.0,0.0,246
126,4544,Administração,PE,1.0,0.0,39
58,176,Pedagogia,PE,0.83871,0.032258,31
34,1582,Administração,PE,0.778756,0.221244,322
59,1771,Administração,PE,0.666667,0.0,72
3,1034,Pedagogia,PE,0.630352,0.077507,270
35,1582,Pedagogia,PE,0.61219,0.38781,145
100,375,Administração,PE,0.597222,0.041667,72
70,1992,Administração,PE,0.363508,0.158327,370


## 8. Indicador de mercado de trabalho (RAIS) — salário de nível inicial


In [5]:
# curso -> código(s) CBO validado(s). Ciências Sociais usa a família inteira (2511),
# pois não há um código claramente dominante como nos demais cursos.
CURSO_CBO = {
    "Administração": ["252105"],       # Administrador
    "Engenharia Civil": ["214205"],    # Engenheiro civil (maioria dos vínculos da família)
    "Direito": ["241005"],             # Advogado (maior subgrupo da família 2410)
    "Pedagogia": ["231210"],           # Professor de nível superior / pedagogo (código validado)
    "Ciências Sociais": ["251105", "251110", "251115", "251120"],  # família 2511 completa
}

ANO_RAIS = 2022
IDADE_MAX_NIVEL_INICIAL = 30  # aproxima "recém-formado" — ver nota acima

# Amostra mínima de vínculos abaixo da qual o resultado deve ser lido com ressalva.
AMOSTRA_MINIMA_CONFIAVEL = 30

linhas_mercado = []
usou_rais_real = False

try:
    for curso, codigos in CURSO_CBO.items():
        codigos_sql = ", ".join([f"'{c}'" for c in codigos])
        query_rais = f"""
        SELECT
          SUM(valor_remuneracao_media) / COUNT(*) AS salario_medio_inicial,
          COUNT(*) AS qtd_vinculos_amostra
        FROM `basedosdados.br_me_rais.microdados_vinculos`
        WHERE ano = {ANO_RAIS}
          AND sigla_uf = '{UF}'
          AND cbo_2002 IN ({codigos_sql})
          AND valor_remuneracao_media > 0
          AND idade <= {IDADE_MAX_NIVEL_INICIAL}
        """
        df_r = bd.read_sql(query_rais, billing_project_id=PROJECT_ID)
        valor = df_r["salario_medio_inicial"].iloc[0]
        qtd = int(df_r["qtd_vinculos_amostra"].iloc[0])

        if pd.notna(valor) and valor > 0:
            linhas_mercado.append({
                "nome_curso": curso,
                "salario_medio_area": round(float(valor), 2),
                "qtd_vinculos_amostra": qtd,
            })
        else:
            raise ValueError(f"RAIS retornou vazio/zero para {curso}")

    df_mercado = pd.DataFrame(linhas_mercado)
    usou_rais_real = True
    print("Indicador de mercado obtido do RAIS real (salário de nível inicial, idade <= 30):")

except Exception as e:
    print(f"Não deu pra usar o RAIS real ({e}). Caindo para o Plano B manual.")

    # Plano B: valores já validados na consulta real feita anteriormente (deixe aqui
    # como referência / fallback caso a query em ambiente novo não rode a tempo).
    indicador_mercado_manual = {
        "Administração": (4354.88, 2208),
        "Engenharia Civil": (6849.80, 711),
        "Direito": (3322.46, 563),
        "Pedagogia": (2023.69, 2670),
        "Ciências Sociais": (3361.73, 15),
    }
    df_mercado = pd.DataFrame([
        {"nome_curso": c, "salario_medio_area": v[0], "qtd_vinculos_amostra": v[1]}
        for c, v in indicador_mercado_manual.items()
    ])

# Aviso automático de amostra pequena — evita depender de lembrar manualmente
# de flagar isso nos slides.
df_mercado["amostra_pequena"] = df_mercado["qtd_vinculos_amostra"] < AMOSTRA_MINIMA_CONFIAVEL

for _, linha in df_mercado.iterrows():
    if linha["amostra_pequena"]:
        print(
            f"⚠️  Aviso: {linha['nome_curso']} tem amostra pequena de nível inicial "
            f"({linha['qtd_vinculos_amostra']} vínculos, abaixo do mínimo de "
            f"{AMOSTRA_MINIMA_CONFIAVEL}) — resultado deve ser lido com ressalva "
            f"(citar isso explicitamente na apresentação)."
        )

print(f"usou_rais_real = {usou_rais_real}")
df_mercado


Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Indicador de mercado obtido do RAIS real (salário de nível inicial, idade <= 30):
⚠️  Aviso: Ciências Sociais tem amostra pequena de nível inicial (15 vínculos, abaixo do mínimo de 30) — resultado deve ser lido com ressalva (citar isso explicitamente na apresentação).
usou_rais_real = True


,nome_curso,salario_medio_area,qtd_vinculos_amostra,amostra_pequena
0,Administração,4354.88,2208,False
1,Engenharia Civil,6849.80,711,False
2,Direito,3322.46,563,False
3,Pedagogia,2023.69,2670,False
4,Ciências Sociais,3361.73,15,True


## 9. Aprender o peso do indicador de mercado sobre a evasão


In [6]:
from sklearn.linear_model import LinearRegression

df_final_curso = df_evasao.merge(df_mercado, on="nome_curso", how="left")

X = df_final_curso[["salario_medio_area"]].values
y = df_final_curso["taxa_evasao_real"].values

modelo = LinearRegression().fit(X, y)

coeficiente_salario = modelo.coef_[0]
intercepto = modelo.intercept_

print(f"Coeficiente (efeito do salário de nível inicial sobre a evasão real): {coeficiente_salario:.6f}")
print(f"Intercepto: {intercepto:.4f}")
print(
    "Leitura: para cada R$ 1.000 a mais de salário de nível inicial na área, a evasão real muda em "
    f"{coeficiente_salario * 1000:.4f} (em pontos percentuais de 0 a 1)."
)


Coeficiente (efeito do salário de nível inicial sobre a evasão real): -0.000005
Intercepto: 0.0879
Leitura: para cada R$ 1.000 a mais de salário de nível inicial na área, a evasão real muda em -0.0046 (em pontos percentuais de 0 a 1).


## 10. Função do score explicável


- **Evasão histórica**
- **Empregabilidade**
- **Desempenho Academico**
- **Renda Per Capita**

In [7]:
import numpy as np

SM_2026 = 1621  # salário mínimo de referência (2026)
TETO_SALARIO_INICIAL = 7000  #salário de nível inicial

def calcular_score(persona, df_evasao=df_evasao, df_mercado=df_mercado,
                    coef_salario=coeficiente_salario):
    """
    persona: dict com 'nome_curso', 'id_ies' (opcional), 'renda_per_capita',
             'desempenho_academico' (0 a 100)
    """
    score = 50  # base neutra
    composicao = []

    # 1) Evasão real do curso
    if persona.get("id_ies"):
        linha = df_evasao[
            (df_evasao["nome_curso"] == persona["nome_curso"]) &
            (df_evasao["id_ies"] == persona["id_ies"])
        ]
    else:
        linha = df_evasao[df_evasao["nome_curso"] == persona["nome_curso"]]

    taxa_evasao = linha["taxa_evasao_real"].mean()
    pontos_evasao = -round(30 * taxa_evasao)
    score += pontos_evasao
    composicao.append(
        f"Evasão real do curso, janela estendida ({taxa_evasao:.1%}): {pontos_evasao:+d} pontos"
    )

    # 2) Empregabilidade (0 a +20 pontos, entre 1 salário mínimo e o teto de nível inicial)
    linha_mercado = df_mercado.loc[df_mercado["nome_curso"] == persona["nome_curso"]]
    salario_area = linha_mercado["salario_medio_area"].values[0]
    amostra_pequena = bool(linha_mercado["amostra_pequena"].values[0]) if "amostra_pequena" in linha_mercado else False

    frac_salario = np.clip((salario_area - SM_2026) / (TETO_SALARIO_INICIAL - SM_2026), 0, 1)
    pontos_mercado = round(20 * frac_salario)
    score += pontos_mercado
    aviso_amostra = "  ⚠️ amostra pequena de RAIS para este curso — ler com ressalva" if amostra_pequena else ""
    composicao.append(
        f"Empregabilidade da área (salário nível inicial R$ {salario_area:,.0f}): "
        f"{pontos_mercado:+d} pontos{aviso_amostra}"
    )

    # 3) Desempenho acadêmico do candidato (até ±20 pontos, 50 = neutro)
    pontos_desempenho = round((persona["desempenho_academico"] - 50) * 0.4)
    score += pontos_desempenho
    composicao.append(
        f"Desempenho acadêmico ({persona['desempenho_academico']}): {pontos_desempenho:+d} pontos"
    )

    # 4) Renda per capita (-20 a 0 pontos, usando os limiares oficiais do FIES)
    limiar_inferior = 0.5 * SM_2026   # R$ 810,50 — corte do FIES Social
    limiar_superior = 3 * SM_2026     # R$ 4.863 — teto do FIES regular
    frac_acima_piso = np.clip(
        (persona["renda_per_capita"] - limiar_inferior) / (limiar_superior - limiar_inferior), 0, 1
    )
    pontos_renda = -round(20 * (1 - frac_acima_piso))
    score += pontos_renda
    composicao.append(f"Renda per capita (R$ {persona['renda_per_capita']}): {pontos_renda:+d} pontos")

    score_bruto = score
    score = max(0, min(100, score))  # limita a escala a 0-100 para leitura mais intuitiva
    if score != score_bruto:
        composicao.append(f"(score bruto era {score_bruto}, ajustado para a escala 0-100)")

    return score, composicao


def imprimir_parecer(nome_persona, persona):
    score, composicao = calcular_score(persona)
    print(f"=== {nome_persona} ===")
    print(f"Curso: {persona['nome_curso']}")
    for linha in composicao:
        print(f"  {linha}")
    print(f"  SCORE FINAL: {score} pontos\n")


## 11. Rodar o score em personas de exemplo


In [8]:
personas = {
    "Persona A": {
        "nome_curso": "Engenharia Civil",
        "renda_per_capita": 1800,
        "desempenho_academico": 78,
    },
    "Persona B": {
        "nome_curso": "Administração",
        "renda_per_capita": 900,
        "desempenho_academico": 60,
    },
    "Persona C": {
        "nome_curso": "Direito",
        "renda_per_capita": 2500,
        "desempenho_academico": 85,
    },
    "Persona D (fronteira: renda muito baixa + alta empregabilidade)": {
        "nome_curso": "Engenharia Civil",
        "renda_per_capita": 400,   # bem abaixo do corte do FIES Social (0,5 SM)
        "desempenho_academico": 82,
    },
    "Persona E (fronteira: renda alta + curso de maior evasão real)": {
        "nome_curso": "Ciências Sociais",
        "renda_per_capita": 6000,
        "desempenho_academico": 55,
    },
    "Persona F (Pedagogia)": {
        "nome_curso": "Pedagogia",
        "renda_per_capita": 1200,
        "desempenho_academico": 75,
    },
    "Persona G (Ciências Sociais — amostra pequena de RAIS)": {
        "nome_curso": "Ciências Sociais",
        "renda_per_capita": 1500,
        "desempenho_academico": 70,
    },
}

for nome, persona in personas.items():
    imprimir_parecer(nome, persona)


=== Persona A ===
Curso: Engenharia Civil
  Evasão real do curso, janela estendida (1.5%): +0 pontos
  Empregabilidade da área (salário nível inicial R$ 6,850): +19 pontos
  Desempenho acadêmico (78): +11 pontos
  Renda per capita (R$ 1800): -15 pontos
  SCORE FINAL: 65 pontos

=== Persona B ===
Curso: Administração
  Evasão real do curso, janela estendida (9.6%): -3 pontos
  Empregabilidade da área (salário nível inicial R$ 4,355): +10 pontos
  Desempenho acadêmico (60): +4 pontos
  Renda per capita (R$ 900): -20 pontos
  SCORE FINAL: 41 pontos

=== Persona C ===
Curso: Direito
  Evasão real do curso, janela estendida (0.0%): +0 pontos
  Empregabilidade da área (salário nível inicial R$ 3,322): +6 pontos
  Desempenho acadêmico (85): +14 pontos
  Renda per capita (R$ 2500): -12 pontos
  SCORE FINAL: 58 pontos

=== Persona D (fronteira: renda muito baixa + alta empregabilidade) ===
Curso: Engenharia Civil
  Evasão real do curso, janela estendida (1.5%): +0 pontos
  Empregabilidade da ár